<a href="https://colab.research.google.com/github/pranavchauhann/sentiment-finetuning/blob/main/sentiment_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Inference with Fine-Tuned DistilBERT

This notebook uses our fine-tuned DistilBERT model from Hugging Face Hub to predict the sentiment of new movie reviews.

The output will include:

- Predicted sentiment: Positive or Negative
- Confidence score

## Loading the Fine-Tuned Model

We will load our previously fine-tuned DistilBERT model and tokenizer directly from Hugging Face Hub.

No retraining is required.

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "pranavchauhann/sentiment-distilbert-imdb"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


## Creating the Prediction Function

This function will:

1. Take a movie review as input
2. Tokenize the text
3. Pass it through the fine-tuned model
4. Convert logits into probabilities
5. Return the predicted sentiment and confidence score

In [2]:
import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)

    predicted_class = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_class].item()

    sentiment = "Positive" if predicted_class == 1 else "Negative"

    return sentiment, confidence

## Testing the Prediction Function

Now we will test the function with a new movie review.

The function should return:
- predicted sentiment
- confidence score

In [3]:
review = "This movie was absolutely amazing. I loved every minute of it!"

sentiment, confidence = predict_sentiment(review)

print("Review:", review)
print("Sentiment:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")

Review: This movie was absolutely amazing. I loved every minute of it!
Sentiment: Positive
Confidence: 99.59 %


## Testing a Negative Review

Now we will test the same prediction function with a clearly negative movie review.

In [4]:
review = "This movie was boring, badly written, and a complete waste of time."

sentiment, confidence = predict_sentiment(review)

print("Review:", review)
print("Sentiment:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")

Review: This movie was boring, badly written, and a complete waste of time.
Sentiment: Negative
Confidence: 99.74 %


## Interactive Sentiment Prediction

Now the user can type any movie review.

The model will automatically predict:
- Sentiment
- Confidence score

In [5]:
review = input("Enter a movie review: ")

sentiment, confidence = predict_sentiment(review)

print("\nPrediction:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")

Enter a movie review: The story was good but the ending was disappointing.

Prediction: Negative
Confidence: 98.82 %


### Observation on Mixed Sentiment

The review contained both positive and negative opinions.

The model predicted **Negative** with high confidence.

This shows that confidence represents the model's certainty, not guaranteed correctness. Mixed or nuanced reviews can still be challenging for binary sentiment models.

## Predicting Multiple Reviews

Now we will predict sentiment for multiple movie reviews at once.

This is called batch inference.

Instead of processing one review at a time, the tokenizer can process a list of reviews together.

In [6]:
reviews = [
    "This movie was amazing and beautifully made.",
    "The movie was boring and a complete waste of time.",
    "The acting was good, but the story was disappointing."
]

inputs = tokenizer(
    reviews,
    return_tensors="pt",
    padding=True,
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

predicted_classes = torch.argmax(probabilities, dim=1)

## Displaying Batch Predictions

Now we will display the prediction and confidence score for each review in the batch.

In [7]:
for i, review in enumerate(reviews):
    predicted_class = predicted_classes[i].item()
    confidence = probabilities[i][predicted_class].item()

    sentiment = "Positive" if predicted_class == 1 else "Negative"

    print(f"Review {i+1}: {review}")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence * 100:.2f}%")
    print("-" * 50)

Review 1: This movie was amazing and beautifully made.
Sentiment: Positive
Confidence: 99.65%
--------------------------------------------------
Review 2: The movie was boring and a complete waste of time.
Sentiment: Negative
Confidence: 99.73%
--------------------------------------------------
Review 3: The acting was good, but the story was disappointing.
Sentiment: Negative
Confidence: 99.21%
--------------------------------------------------
